# Notebook 7 — photon-number versus indivisible energy packets

Level-1 code: the same cascade under two bookkeepings. A packet absorbed at 455 nm carries energy $E$. Photon-number bookkeeping keeps the *number* of photons and gives them the exit line's frequency, so $E_{\rm out} = w\,h\nu' \neq E$. Indivisible energy packets keep $E$ and let the photon number change. This is the chapter that explains F50.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[0] / "src"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rtedu
from rtedu import results
from rtedu.visualization import save_fig, OI
from rtedu.atom import five_level_atom, H_ERG_S
rng = np.random.default_rng(rtedu.SEEDS["ch07"])
atom = five_level_atom()
T, n_total, t = 4000.0, 30.0, 2.0 * rtedu.DAY        # the book's standard state from here on
r_out = 0.2 * rtedu.C * t
tau = atom.line_list(T, n_total, t); emis = atom.thermal_emissivity(T, n_total)
nm = 1e7 * atom.lam_cm

In [ ]:
from rtedu.macroatom import ToyMacroAtom
E = 1.0                                             # the packet's energy (arbitrary units)
nu_abs = atom.nu[0]                                 # absorbed in 455 nm (4 -> 0): activated at E4
w = E / (H_ERG_S * nu_abs)                          # the photon number the packet represents

def one_activation(rng, mode):
    """the exit line of one cascade-like walk (chapter 8's rule), under one bookkeeping"""
    exit_line, _ = ToyMacroAtom(atom).walk(rng, 4)
    nu_out = atom.nu[exit_line]
    if mode == "energy":
        return exit_line, E, E / (H_ERG_S * nu_out)             # E kept, w changes
    return exit_line, w * H_ERG_S * nu_out, w                   # w kept, E changes

n = 20_000
out = {}
for mode in ("photon", "energy"):
    lines = np.empty(n, int); E_out = np.empty(n)
    for i in range(n):
        lines[i], E_out[i], _ = one_activation(rng, mode)
    out[mode] = dict(E_out_mean=float(E_out.mean()), E_out_min=float(E_out.min()), E_out_max=float(E_out.max()),
                     per_line=np.bincount(lines, weights=E_out, minlength=atom.n_lines) / n)
    print(f"{mode:>6s} packets: <E_out>/E_in = {E_out.mean():.3f}  (min {E_out.min():.3f}, max {E_out.max():.3f})")

The difference is not small: the photon-number scheme loses on average the fraction of the energy that the cascade puts into the photons it drops. Which lines carry the loss:

In [ ]:
deficit = out["energy"]["per_line"] - out["photon"]["per_line"]
for k in range(atom.n_lines):
    if abs(deficit[k]) > 1e-3:
        print(f"line {k} ({nm[k]:.0f} nm): energy {out['energy']['per_line'][k]:.3f}  photon-number {out['photon']['per_line'][k]:.3f}")

## Validation against `rtedu`

In [ ]:
ex_e, E_e, _ = ToyMacroAtom(atom, mode="energy").run(np.random.default_rng(1), 4, 2000, E_in=E)
ex_p, E_p, _ = ToyMacroAtom(atom, mode="photon").run(np.random.default_rng(1), 4, 2000, E_in=E)
assert np.all(E_e == E) and abs(E_p.mean() - out["photon"]["E_out_mean"]) < 0.02
print("rtedu energy mode conserves E exactly; photon mode <E_out> =", round(float(E_p.mean()), 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
x = np.arange(atom.n_lines); w_ = 0.38
axes[0].bar(x - w_ / 2, out["energy"]["per_line"], w_, color=OI["blue"], label="indivisible energy packets")
axes[0].bar(x + w_ / 2, out["photon"]["per_line"], w_, color=OI["red"], label="photon-number packets")
axes[0].set_xticks(x); axes[0].set_xticklabels([f"{v:.0f}" for v in nm], rotation=60, fontsize=7); axes[0].set_ylabel("energy out per activation / E in"); axes[0].legend(fontsize=8)
axes[0].set_title("the same walks, two bookkeepings", fontsize=9)
axes[1].bar(["energy packets", "photon-number packets"], [out["energy"]["E_out_mean"], out["photon"]["E_out_mean"]], color=[OI["blue"], OI["red"]])
axes[1].axhline(1.0, color="k", lw=0.8); axes[1].set_ylabel(r"$\langle E_{\rm out}\rangle / E_{\rm in}$"); axes[1].set_title("energy conservation", fontsize=9)
fig.tight_layout(); save_fig(fig, "ch07_energy_packets")

In [ ]:
results.record("ch07", dict(n=n, absorbed_nm=1e7 * rtedu.C / nu_abs, lines_nm=nm,
                            energy=dict(E_out_mean=out["energy"]["E_out_mean"], per_line=out["energy"]["per_line"]),
                            photon=dict(E_out_mean=out["photon"]["E_out_mean"], E_out_min=out["photon"]["E_out_min"], E_out_max=out["photon"]["E_out_max"], per_line=out["photon"]["per_line"]),
                            deficit_percent=100 * (1 - out["photon"]["E_out_mean"])))